# 03 — How QAOA responds to quantum noise

**Phase 3 | Controlled noise-sensitivity benchmark**

This notebook takes the **frozen best parameters** from Phase 2 and measures how their solution quality changes when we simulate errors on the circuit's CX gates, and optionally readout errors. It does **not** retrain parameters or claim hardware-realistic calibration. We retain exact classical Max-Cut solutions as the comparison target.

**Primary question:** Does the extra circuit depth of QAOA $p=2$ remain helpful under simulated noise?

### 1. Load libraries and locate the Phase 2 results

Reuse your existing environment and seed **31**. This notebook contains all computation code locally; it reads only the saved Phase 2 experiment manifest and its CSV for input data. No project-internal imports are needed.

In [ ]:
from pathlib import Path
from hashlib import sha256
import json
import platform
import time

import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd
import qiskit
import qiskit_aer
from qiskit import QuantumCircuit, transpile
from qiskit.quantum_info import Statevector
from qiskit_aer import AerSimulator
from qiskit_aer.noise import NoiseModel, ReadoutError, depolarizing_error
from IPython.display import display

SEED = 31
SHOTS = 4096
REPEATS = 2
CX_ERROR_LEVELS = (0.0, 0.005, 0.02, 0.05, 0.10)
NOISE_MODES = ('cx_only', 'cx_plus_readout')

ROOT = Path.cwd()
if ROOT.name.lower() == 'notebooks':
    ROOT = ROOT.parent
assert (ROOT / 'notebooks').exists(), f'Open this notebook from inside QuantumOptLab: {ROOT}'
EXP = ROOT / 'reports' / 'experiments'
FIG = ROOT / 'reports' / 'figures'
EXP.mkdir(parents=True, exist_ok=True)
FIG.mkdir(parents=True, exist_ok=True)
PHASE02_MANIFEST = EXP / 'phase02_manifest_seed31.json'
PHASE02_RESULTS = EXP / 'phase02_qaoa_optimization_seed31.csv'
assert PHASE02_MANIFEST.exists() and PHASE02_RESULTS.exists(), (
    'Missing Phase 2 results. Run all cells in 02_maxcut_qaoa_baselines.ipynb first.'
)
print('Qiskit', qiskit.__version__, '| Aer', qiskit_aer.__version__)
print('Project:', ROOT)
print('Synthetic CX depolarizing levels:', CX_ERROR_LEVELS)
print('Shots per circuit:', SHOTS, '| Independent simulation seeds:', REPEATS)

### 2. Restore the exact Max-Cut problems and trained parameters

Read the original graph definitions and parameters from Phase 2. Represent bitstring $x$ as an integer, with qubit 0 at the right of Qiskit's displayed bitstring. Reconstruct exact optimal objective values for all possible assignments.

In [ ]:
phase02 = json.loads(PHASE02_MANIFEST.read_text(encoding='utf-8'))
assert phase02['seed'] == SEED and phase02['algorithm'] == 'statevector-optimized QAOA'
old_results = pd.read_csv(PHASE02_RESULTS)
GRAPHS = {}
COSTS = {}
OPTIMA = {}
PARAMS = {}
for name, edges in phase02['graph_edges'].items():
    g = nx.Graph()
    g.add_weighted_edges_from([(int(u), int(v), float(w)) for u, v, w in edges])
    g = nx.convert_node_labels_to_integers(g, ordering='sorted')
    assert nx.is_connected(g)
    GRAPHS[name] = g
    n_vertices = g.number_of_nodes()
    costs = np.array([
        sum(w for u, v, w in g.edges.data('weight') if ((assignment >> u) & 1) != ((assignment >> v) & 1))
        for assignment in range(1 << n_vertices)
    ], dtype=float)
    COSTS[name] = costs
    OPTIMA[name] = float(costs.max())
    for depth in (1, 2):
        key = f'{name}__p{depth}'
        PARAMS[name, depth] = np.asarray(phase02['best_parameters'][key], dtype=float)
        assert PARAMS[name, depth].shape == (2 * depth,)

baseline_table = pd.DataFrame([{
    'graph': name, 'qubits': g.number_of_nodes(), 'edges': g.number_of_edges(),
    'uniform_random_mean': COSTS[name].mean(), 'exact_max_cut': OPTIMA[name]
} for name, g in GRAPHS.items()])
display(baseline_table)

### 3. Reproduce the ideal Phase 2 QAOA outputs

This independent reconstruction is an important cross-phase integrity test. It confirms that the graph-to-circuit mapping and saved parameters recover Phase 2's optimized expected objective. We evaluate the statevector exactly, without shot noise.

In [ ]:
def qaoa_circuit(graph, params, depth):
    n_qubits = graph.number_of_nodes()
    assert len(params) == 2 * depth
    qc = QuantumCircuit(n_qubits)
    qc.h(range(n_qubits))
    for gamma, beta in zip(params[:depth], params[depth:]):
        for u, v, data in graph.edges(data=True):
            qc.rzz(-float(gamma) * float(data['weight']), u, v)
        for qubit in range(n_qubits):
            qc.rx(2 * float(beta), qubit)
    return qc

IDEAL = {}
for (name, depth), params in PARAMS.items():
    probs = np.abs(Statevector.from_instruction(qaoa_circuit(GRAPHS[name], params, depth)).data) ** 2
    expected_cut = float(probs @ COSTS[name])
    probability_optimal = float(probs[np.isclose(COSTS[name], OPTIMA[name])].sum())
    original = old_results.loc[(old_results['graph'] == name) & (old_results['depth_p'] == depth)].iloc[0]
    assert np.isclose(expected_cut, original['qaoa_expected_cut'], atol=5e-4), (name, depth)
    assert np.isclose(probability_optimal, original['probability_optimal_bitstring'], atol=5e-4)
    IDEAL[name, depth] = {'expected_cut': expected_cut, 'p_optimal': probability_optimal}

print('PASS: All six saved Phase 2 QAOA solutions reproduce within CSV rounding tolerance.')
display(pd.DataFrame([{'graph': name, 'p': depth, **stats} for (name, depth), stats in IDEAL.items()]).round(4))

### 4. Compile to the gates that actually receive noise

Qiskit's high-level `rzz` operation would not receive CX noise unless we first decompose it. Compile into `h`, `rx`, `rz`, and `cx` at optimization level **0** to preserve a consistent, auditable circuit. Record how many CX gates each circuit contains. This study uses unrestricted all-to-all connectivity and no hardware-specific gate durations.

In [ ]:
COMPILED = {}
resources = []
for (name, depth), params in PARAMS.items():
    qc = qaoa_circuit(GRAPHS[name], params, depth)
    compiled = transpile(qc, basis_gates=['h', 'rx', 'rz', 'cx'],
                         optimization_level=0, seed_transpiler=SEED)
    gate_counts = dict(compiled.count_ops())
    assert 'rzz' not in gate_counts, f'Undecomposed rzz gate for {name} p={depth}'
    cx_count = int(gate_counts.get('cx', 0))
    assert cx_count == 2 * GRAPHS[name].number_of_edges() * depth, (name, depth, gate_counts)
    compiled.measure_all()
    COMPILED[name, depth] = compiled
    resources.append({'graph': name, 'p': depth, 'cx_gates': cx_count,
                      'measured_circuit_depth': compiled.depth(), 'total_ops': sum(compiled.count_ops().values())})
resource_df = pd.DataFrame(resources)
display(resource_df)
print('PASS: Every Ising ZZ interaction was decomposed into two CX gates per QAOA layer.')

### 5. Define controlled noise and evaluation metrics

Use the exact same saved parameter values for all noise levels. In `cx_only` mode, apply a two-qubit depolarizing channel **after each CX**. In `cx_plus_readout` mode, additionally flip each measured bit independently with probability $q=p_{\mathrm{CX}}/2$. This coupling is an experimental convention, **not** an IBM device calibration.

Track (a) sampled expected cut divided by the exact optimum and (b) frequency of measuring a globally optimal bitstring. Shot randomness is repeated with independent seeds.

In [ ]:
def make_noise_model(cx_error, readout_error):
    model = NoiseModel()
    if cx_error > 0:
        model.add_all_qubit_quantum_error(depolarizing_error(float(cx_error), 2), ['cx'])
    if readout_error > 0:
        q = float(readout_error)
        model.add_all_qubit_readout_error(ReadoutError([[1-q, q], [q, 1-q]]))
    return model


def evaluate_counts(name, counts):
    costs = COSTS[name]
    optimum = OPTIMA[name]
    shots = sum(counts.values())
    values = [(costs[int(bits.replace(' ', ''), 2)], count) for bits, count in counts.items()]
    expected = float(sum(value * count for value, count in values) / shots)
    p_optimal = float(sum(count for value, count in values if np.isclose(value, optimum)) / shots)
    return expected, p_optimal

# Two informative checks on the noise model itself.
assert not make_noise_model(0.0, 0.0).noise_instructions
assert 'cx' in make_noise_model(0.02, 0.0).noise_instructions
assert 'measure' in make_noise_model(0.0, 0.01).noise_instructions
print('PASS: Noise channel attachment matches the intended gates and measurement operations.')

### 6. Run the noise-sensitivity experiments

For each graph and circuit depth, evaluate five CX error strengths under both noise modes, repeating the final measurements twice with different seeds. **No optimizer calls occur here**: this is inference-time robustness of the original Phase 2 solutions. Each run is recorded separately for reproducibility.

In [ ]:
raw_rows = []
run_begin = time.perf_counter()
for graph_index, name in enumerate(GRAPHS):
    for level_index, cx_error in enumerate(CX_ERROR_LEVELS):
        for mode_index, mode in enumerate(NOISE_MODES):
            readout_error = cx_error / 2 if mode == 'cx_plus_readout' else 0.0
            model = make_noise_model(cx_error, readout_error)
            simulator = AerSimulator(method='density_matrix', noise_model=model)
            for depth in (1, 2):
                circuit = COMPILED[name, depth]
                for repeat in range(REPEATS):
                    sim_seed = SEED + 10000 * graph_index + 1000 * mode_index + 100 * level_index + repeat
                    result = simulator.run(circuit, shots=SHOTS, seed_simulator=sim_seed).result()
                    assert result.success, f'Aer simulation failed: {name}, p={depth}, cx={cx_error}'
                    counts = result.get_counts()
                    assert sum(counts.values()) == SHOTS
                    mean_cut, p_optimal = evaluate_counts(name, counts)
                    raw_rows.append({
                        'graph': name, 'depth_p': depth, 'noise_mode': mode,
                        'cx_depolarizing_parameter': cx_error, 'readout_flip_probability': readout_error,
                        'repeat': repeat, 'seed_simulator': sim_seed, 'shots': SHOTS,
                        'sampled_expected_cut': mean_cut, 'approximation_ratio': mean_cut / OPTIMA[name],
                        'p_optimal': p_optimal, 'exact_max_cut': OPTIMA[name],
                        'ideal_statevector_expected_cut': IDEAL[name, depth]['expected_cut'],
                        'ideal_statevector_p_optimal': IDEAL[name, depth]['p_optimal'],
                    })
    print(f'Completed: {name} ({len(CX_ERROR_LEVELS)} CX noise levels × {len(NOISE_MODES)} modes × 2 depths × {REPEATS} repeats)')
raw_df = pd.DataFrame(raw_rows)
print(f'Completed {len(raw_df)} circuit simulations in {time.perf_counter()-run_begin:.1f}s')
display(raw_df.head(8).round(4))

### 7. Aggregate the measurements without hiding uncertainty

Report the mean and standard deviation across independently seeded sampling runs. These standard deviations describe **finite-shot variability**, not variation across graphs or calibration uncertainty. We do not claim statistical significance or hardware realism on this small benchmark.

In [ ]:
keys = ['graph', 'depth_p', 'noise_mode', 'cx_depolarizing_parameter', 'readout_flip_probability']
summary_df = raw_df.groupby(keys, as_index=False).agg(
    mean_approx_ratio=('approximation_ratio', 'mean'),
    std_approx_ratio=('approximation_ratio', 'std'),
    mean_optimal_probability=('p_optimal', 'mean'),
    std_optimal_probability=('p_optimal', 'std'),
    mean_cut=('sampled_expected_cut', 'mean'),
    n_repeats=('repeat', 'count'),
)
summary_df['std_approx_ratio'] = summary_df['std_approx_ratio'].fillna(0.0)
summary_df['std_optimal_probability'] = summary_df['std_optimal_probability'].fillna(0.0)
assert len(summary_df) == len(GRAPHS) * 2 * len(NOISE_MODES) * len(CX_ERROR_LEVELS)
focus = summary_df[(summary_df.noise_mode == 'cx_only') &
                   (summary_df.cx_depolarizing_parameter.isin([0.0, 0.02, 0.10]))]
display(focus[['graph', 'depth_p', 'cx_depolarizing_parameter', 'mean_approx_ratio',
               'mean_optimal_probability', 'std_approx_ratio']].round(4))

### 8. Plot the sensitivity curves

Plot approximation ratio for depth 1 and 2 on each graph. The ideal p=1 and p=2 reference values are marked as dotted horizontal lines, and classical exact optimization remains the reference ratio **1.0**. Separate figures isolate CX-only noise from added readout noise.

In [ ]:
figure_paths = []
for mode in NOISE_MODES:
    fig, axes = plt.subplots(1, len(GRAPHS), figsize=(11, 3.5), sharey=True)
    for ax, name in zip(axes, GRAPHS):
        for depth in (1, 2):
            rows = summary_df[(summary_df.graph == name) & (summary_df.depth_p == depth) &
                              (summary_df.noise_mode == mode)].sort_values('cx_depolarizing_parameter')
            ax.errorbar(rows.cx_depolarizing_parameter, rows.mean_approx_ratio,
                        yerr=rows.std_approx_ratio, marker='o', capsize=2, label=f'QAOA p={depth}')
        ax.axhline(1.0, color='gray', linestyle=':', linewidth=1, label='Exact maximum')
        ax.set_title(name.replace('_', ' ').title())
        ax.set_xlabel('CX depolarizing parameter')
        ax.set_ylim(0, 1.04)
        ax.grid(alpha=0.2)
    axes[0].set_ylabel('Approximation ratio')
    axes[-1].legend(fontsize=8, loc='lower left')
    fig.suptitle('QAOA Robustness — ' + ('CX Noise Only' if mode == 'cx_only' else 'CX + Readout Noise'), fontsize=12)
    fig.tight_layout()
    path = FIG / f'phase03_qaoa_{mode}_seed31.png'
    fig.savefig(path, dpi=170, bbox_inches='tight')
    plt.show()
    print('Saved:', path)
    figure_paths.append(path)

### 9. Examine the deeper-circuit tradeoff

Compute $\Delta r = r_{p=2}-r_{p=1}$ at every noise level. Positive values favor p=2; negative values favor p=1. This is a **descriptive comparison using observed results**, not a validated automatic circuit selector. To claim an improvement in a later phase, selection must be learned on separate training instances and evaluated on unseen test graphs.

In [ ]:
left = summary_df[summary_df.depth_p == 1].drop(columns=['depth_p'])
right = summary_df[summary_df.depth_p == 2].drop(columns=['depth_p'])
join_keys = ['graph', 'noise_mode', 'cx_depolarizing_parameter', 'readout_flip_probability']
delta_df = left.merge(right, on=join_keys, suffixes=('_p1', '_p2'), validate='one_to_one')
delta_df['p2_minus_p1_ratio'] = delta_df.mean_approx_ratio_p2 - delta_df.mean_approx_ratio_p1
delta_df['observed_favored_depth'] = np.where(delta_df.p2_minus_p1_ratio > 0, 2, 1)
display(delta_df[['graph', 'noise_mode', 'cx_depolarizing_parameter', 'mean_approx_ratio_p1',
                  'mean_approx_ratio_p2', 'p2_minus_p1_ratio', 'observed_favored_depth']].round(4))

fig, ax = plt.subplots(figsize=(8.2, 4.2))
for name in GRAPHS:
    rows = delta_df[(delta_df.graph == name) & (delta_df.noise_mode == 'cx_only')].sort_values('cx_depolarizing_parameter')
    ax.plot(rows.cx_depolarizing_parameter, rows.p2_minus_p1_ratio, marker='o', label=name)
ax.axhline(0, color='gray', linewidth=1, linestyle='--')
ax.set_title('Depth-2 Benefit Under CX Noise')
ax.set_xlabel('CX depolarizing parameter')
ax.set_ylabel('Approximation ratio difference (p=2 minus p=1)')
ax.legend()
ax.grid(alpha=0.2)
fig.tight_layout()
delta_figure_path = FIG / 'phase03_depth_tradeoff_seed31.png'
fig.savefig(delta_figure_path, dpi=170, bbox_inches='tight')
plt.show()
print('Saved:', delta_figure_path)

### 10. Verify and save the reproducible experiment

Checks cover source reproducibility, CX decomposition, data completeness, probability bounds, and the zero-noise control. We allow tolerance in finite-shot estimates: exact statevector values need not equal sampled values exactly. Save individual measurements, aggregate tables, and the configuration manifest.

In [ ]:
zero = summary_df[(summary_df.noise_mode == 'cx_only') &
                  (summary_df.cx_depolarizing_parameter == 0.0)]
zero_differences = [abs(row.mean_cut - IDEAL[row.graph, int(row.depth_p)]['expected_cut']) for row in zero.itertuples()]
checks = {
    'Six Phase 2 parameters reproduced': len(IDEAL) == 6,
    'CX decomposition matches 2 × edges × depth': all(
        int(row.cx_gates) == 2 * GRAPHS[row.graph].number_of_edges() * int(row.p)
        for row in resource_df.itertuples()),
    'All noise simulations recorded': len(raw_df) == len(GRAPHS) * 2 * len(NOISE_MODES) * len(CX_ERROR_LEVELS) * REPEATS,
    'Every circuit uses the requested shots': bool((raw_df.shots == SHOTS).all()),
    'Probability values in [0, 1]': bool(raw_df[['approximation_ratio', 'p_optimal']].ge(-1e-8).all().all()
        and raw_df[['approximation_ratio', 'p_optimal']].le(1 + 1e-8).all().all()),
    'Noise-free sampled means near ideal values': all(diff < 0.20 for diff in zero_differences),
    'Each group has independent repeats': bool((summary_df.n_repeats == REPEATS).all()),
    'Depth tradeoff joins all noise conditions': len(delta_df) == len(GRAPHS) * len(NOISE_MODES) * len(CX_ERROR_LEVELS),
}
for label, passed in checks.items():
    print(('PASS: ' if passed else 'FAIL: ') + label)
assert all(checks.values()), 'Phase 3 verification failed; inspect outputs above.'
print(f'PASS: {sum(checks.values())}/{len(checks)} Phase 3 checks')

outputs = {
    'raw': EXP / 'phase03_qaoa_noise_raw_seed31.csv',
    'summary': EXP / 'phase03_qaoa_noise_summary_seed31.csv',
    'resources': EXP / 'phase03_compiled_gate_resources_seed31.csv',
    'tradeoff': EXP / 'phase03_depth_tradeoff_seed31.csv',
}
raw_df.to_csv(outputs['raw'], index=False)
summary_df.to_csv(outputs['summary'], index=False)
resource_df.to_csv(outputs['resources'], index=False)
delta_df.to_csv(outputs['tradeoff'], index=False)
manifest = {
    'phase': 3, 'seed': SEED, 'shots_per_circuit': SHOTS, 'independent_repeats': REPEATS,
    'qiskit_version': qiskit.__version__, 'aer_version': qiskit_aer.__version__,
    'python_version': platform.python_version(),
    'phase02_manifest_sha256': sha256(PHASE02_MANIFEST.read_bytes()).hexdigest(),
    'cx_error_levels': list(CX_ERROR_LEVELS), 'noise_modes': list(NOISE_MODES),
    'readout_flip_probability': 'cx_error / 2 only in cx_plus_readout',
    'gate_basis': ['h', 'rx', 'rz', 'cx'], 'transpiler_optimization_level': 0,
    'noisy_operation': 'depolarizing_error(p, 2) after every cx',
    'training': 'NONE — original Phase 2 parameters fixed',
    'device': 'synthetic homogeneous, unrestricted-connectivity model, not real hardware',
    'limits': 'Three tiny training instances, only two sampling repeats; exploratory, not a generalizable advantage claim',
}
manifest_path = EXP / 'phase03_manifest_seed31.json'
manifest_path.write_text(json.dumps(manifest, indent=2), encoding='utf-8')
for path in [*outputs.values(), manifest_path]:
    print('Saved:', path)
print('Run completed. Next: expand to held-out graphs and evaluate noise-aware selection out of sample.')